Script for scraping Dutch Articles 

In [1]:
# load the necessary libraries 
import requests
from bs4 import BeautifulSoup
from datetime import datetime
import os 



/usr/lib/python3/dist-packages/requests/__init__.py:89: RequestsDependencyWarning: urllib3 (2.2.3) or chardet (3.0.4) doesn't match a supported version!
  warnings.warn("urllib3 ({}) or chardet ({}) doesn't match a supported "


In [2]:
import os
import platform
os_type = platform.system()

In [ ]:
"""
This script processes a folder of previously downloaded HTML content of CBS articles. 
It parses each HTML file using BeautifulSoup to extract structured content, including:
- the article's title
- publication date
- lead text 
- main article content

If a file is already processed, it is skipped. 
Files missing critical sections or causing errors are logged into a text file (error) for review.
"""


# Fetch the page
html_dir = "downloaded_html_files_nl"
output_dir = "scraped_articles"
error_files = []


html_files = [f for f in os.listdir(html_dir)]



for html_file in html_files:
    file_path = os.path.join(html_dir, html_file)

    output_file_path = os.path.join(output_dir , f"{os.path.splitext(html_file)[0]}.txt")

    if os.path.exists(output_file_path):
        print(f"{output_file_path} already exists. Skipping this file. ")
        continue

    try:
        with open(file_path, "r", encoding="utf-8") as f:
            page_content = f.read()

        soup = BeautifulSoup(page_content, "html.parser")

        # Find the article section
        article = soup.find("article")
        if article:
            # Get date
            date = article.find(class_="date")
            time = date.find("time")
            dt = time.get("datetime")
            dt = datetime.fromisoformat(dt)
            dt = dt.strftime("%Y-%m-%d %H:%M:%S %Z")

            # Get header and title
            header = article.find("header")
            title = header.find("h1").get_text(strip=True)

            # Get the first section lead text
            first_section = article.find_all("section")[1]
            lead_text_content = first_section.get_text()

            # Try to get the second section
            try:
                second_section = article.find_all("section")[2]
            except IndexError:
                # If second section is missing  the file and skip further processing
                print(f"Second section missing in {html_file}")
                error_files.append(html_file)
                continue

            # Clean the second section
            
            for div in second_section.find_all('div'):
                div.decompose()  # Remove all <div> tags


            for script in second_section.find_all('script'):
                script.decompose()  # Remove all <script> tags


            for a in second_section.find_all('a'):
                a.decompose()  # Remove all <a> tags to exclude hrefs

            text_content = second_section.get_text()

            # Remove unwanted text
            text_content = text_content.replace("Made with HC editor version: v1.0.3.15240", "")

            # Save the article text
            
            with open(output_file_path, "w", encoding="utf-8") as out_file:
                out_file.write(f"Title: {title}\n")
                out_file.write(f"Date: {dt}\n\n")
                out_file.write(f"Lead Text:{lead_text_content}\n\n")
                out_file.write(f"Article Content:{text_content}\n\n")

            print(f"File saved: {output_file_path}")

        else:
            print(f"No article section found in {html_file}")
            error_files.append(html_file)

    except Exception as e:
        print(f"Error processing {html_file}: {e}")
        error_files.append(html_file)

#print the list of files with issues
if error_files:
    error_file_path = os.path.join(output_dir, "error_files.txt")
    with open(error_file_path, "w") as error_log:
        for error_file in error_files:
            error_log.write(f"{error_file}\n")
    print(f"\nFiles with issues were logged in: {error_file_path}")
else:
    print("\nNo issues found in the processed files.")



scraped_articles/bezorgdheid-over-internetveiligheid-maakt-mensen-alert.txt already exists. Skipping this file. 
scraped_articles/afzetprijzen-industrie-iets-hoger-in-februari.txt already exists. Skipping this file. 
scraped_articles/baanverlies-in-maart-op-laagtepunt.txt already exists. Skipping this file. 
scraped_articles/sterfte-verder-gedaald-in-week-1.txt already exists. Skipping this file. 
scraped_articles/in-week-52-zijn-32-bedrijven-failliet-verklaard.txt already exists. Skipping this file. 
scraped_articles/opnieuw-minder-mensen-in-de-wettelijke-schuldsanering.txt already exists. Skipping this file. 
scraped_articles/consument-even-positief-als-voorgaande-maanden.txt already exists. Skipping this file. 
scraped_articles/corona-stimuleerde-vooral-in-de-horeca-de-online-verkoop.txt already exists. Skipping this file. 
scraped_articles/economisch-beeld-nagenoeg-onveranderd.txt already exists. Skipping this file. 
scraped_articles/omzet-detailhandel-ruim-3-procent-hoger-in-tweed

In [12]:
import requests
from bs4 import BeautifulSoup


URL = "https://www.cbs.nl/nl-nl/nieuws/2018/31/nagelaten-vermogen-neemt-weer-toe-in-2015"

page = requests.get(URL, verify = False)

/home/iwag@cbsp.nl/.local/lib/python3.8/site-packages/urllib3/connectionpool.py:1099: InsecureRequestWarning: Unverified HTTPS request is being made to host 'webproxy.cbsp.nl'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
